# MP5: Porównanie modeli i rekomendacja końcowa

**MajsterPlus — Predykcja rezygnacji klientów**

W tym mini-projekcie:
1. Wytrenujesz dodatkowe modele (GradientBoosting, opcjonalnie VotingClassifier)
2. Porównasz wszystkie modele pod kątem metryk statystycznych, zysku biznesowego i sprawiedliwości
3. Ocenisz interpretowalność modeli
4. Napiszesz rekomendację końcową dla MajsterPlus

**Faza CRISP-DM**: Ewaluacja (synteza)

---

## 0. Konfiguracja i reprodukowalność

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Weryfikacja wersji bibliotek
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — oczekiwano 1.4.x lub 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — oczekiwano 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — oczekiwano <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Wczytanie checkpoint z MP4

In [ ]:
import hashlib, pickle
from pathlib import Path

# Wykrywanie środowiska Colab
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/PUM/checkpoints")
except ImportError:
    DATA_DIR = Path("../2. data")
    CHECKPOINT_DIR = Path("../checkpoints")

checkpoint_file = CHECKPOINT_DIR / "mp4_checkpoint.pkl"

if checkpoint_file.exists():
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
    print(f"Wczytano checkpoint z MP4: {list(checkpoint.keys())}")
else:
    print(f"Nie znaleziono checkpoint pod ścieżką {checkpoint_file}")
    print("Możesz kontynuować, uruchamiając najpierw MP4 lub wczytując surowe dane.")
    checkpoint = None

In [ ]:
if checkpoint is not None:
    X_train = checkpoint["X_train"]
    X_test = checkpoint["X_test"]
    y_train = checkpoint["y_train"]
    y_test = checkpoint["y_test"]
    feature_names = checkpoint["feature_names"]
    gender_test = checkpoint.get("gender_test")
    lr_model = checkpoint.get("lr_model")
    rf_model = checkpoint.get("rf_model")
    y_prob_lr = checkpoint.get("y_prob_lr")
    y_prob_rf = checkpoint.get("y_prob_rf")
    CAMPAIGN_COST = checkpoint.get("CAMPAIGN_COST", 80)
    EXPECTED_REVENUE = checkpoint.get("EXPECTED_REVENUE")
else:
    raise FileNotFoundError(
        "Nie znaleziono checkpoint. Najpierw uruchom rozwiązanie MP4."
    )

print(f"Wczytano checkpoint z MP4")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Koszt kampanii: {CAMPAIGN_COST} PLN, Oczekiwany przychód: {EXPECTED_REVENUE:.2f} PLN")

## 2. Model 1: LogisticRegression (z MP3)

Już wytrenowany — oblicz predykcje, jeśli nie zostały wczytane z checkpoint.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

y_pred_lr = lr_model.predict(X_test)
if y_prob_lr is None:
    y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

print(f"LogReg ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")

## 3. Model 2: RandomForest (z MP3)

Już wytrenowany — oblicz predykcje, jeśli nie zostały wczytane z checkpoint.

In [ ]:
y_pred_rf = rf_model.predict(X_test)
if y_prob_rf is None:
    y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print(f"RF ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

## 4. Model 3: GradientBoosting

**Zadanie**: Wytrenuj klasyfikator GradientBoosting. Użyj `random_state=42` dla reprodukowalności (domyślne hiperparametry).
Oblicz predykcje twarde (`y_pred_gb`) i prawdopodobieństwa (`y_prob_gb`), a następnie wyświetl ROC-AUC.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# TWÓJ KOD TUTAJ
# Wytrenuj klasyfikator GradientBoosting. Użyj random_state=42 dla reprodukowalności (domyślne hiperparametry).
# gb = ...
# y_pred_gb = ...
# y_prob_gb = ...
# Wyświetl ROC-AUC

## 5. Model 4: VotingClassifier (opcjonalny)

**Zadanie** (opcjonalne): Stwórz zespół z głosowaniem miękkim (soft-voting), który uśrednia przewidywane prawdopodobieństwa z LogReg, RF i GB.
Użyj `voting="soft"` i uwzględnij wszystkie trzy wcześniej wytrenowane modele jako estymatory.

In [ ]:
from sklearn.ensemble import VotingClassifier

# TWÓJ KOD TUTAJ (opcjonalny, ale zalecany)
# Stwórz zespół z głosowaniem miękkim z trzech pojedynczych modeli.
# voting = ...
# y_pred_vc = ...
# y_prob_vc = ...
# Wyświetl ROC-AUC

## 6. Tabela porównawcza (wiele kryteriów)

**Zadanie**: Zbuduj DataFrame porównawczy z jednym wierszem na model i następującymi kolumnami:
Model, Accuracy, Precision, Recall, F1, ROC-AUC, Zysk (PLN).

Zysk powinien być obliczony przy domyślnym threshold (0.5) za pomocą zdefiniowanej poniżej funkcji `compute_profit`.

Funkcja `compute_profit` przyjmuje `y_true`, `y_pred`, `revenue` i `cost` i zwraca:
```
TP * (revenue - cost) + FP * (-cost)
```

In [ ]:
# TWÓJ KOD TUTAJ
# 1. Zdefiniuj compute_profit(y_true, y_pred, revenue, cost)
# 2. Zbuduj słownik modeli: {nazwa: (y_pred, y_prob), ...}
# 3. Przejdź po modelach i oblicz wszystkie metryki + zysk
# 4. Stwórz DataFrame i wyświetl go

### Zastanów się

**Kluczowa obserwacja**: Sprawdź, który model ma najwyższy ROC-AUC, a który najwyższy zysk. Czy to ten sam model? Jeśli nie, dlaczego?

**TODO**: Wyjaśnij własnymi słowami, dlaczego model z lepszą ogólną zdolnością dyskryminacyjną (AUC) może nie generować najwyższego zysku przy określonym threshold.

In [ ]:
# TWOJA ODPOWIEDŹ TUTAJ
# Napisz swoje wyjaśnienie jako instrukcje print() lub komentarze.
# Rozważ: Co mierzy AUC w skali? Od czego zależy zysk?

## 7. Porównanie zysku biznesowego

**Zadanie**: Dla każdego modelu przetestuj threshold od 0.05 do 0.95 (krok 0.05) i oblicz zysk przy każdym threshold.
Znajdź optymalny threshold i maksymalny zysk dla każdego modelu. Podaj również zysk przy domyślnym threshold (0.5).

Narysuj wykres zysku w funkcji threshold dla wszystkich modeli na jednym wykresie.

In [ ]:
# TWÓJ KOD TUTAJ
# thresholds = np.arange(0.05, 1.0, 0.05)
# Dla każdego modelu:
#   - Oblicz zysk przy threshold 0.5
#   - Przetestuj wszystkie threshold, znajdź optymalny
# Wyświetl wyniki i stwórz wykres zysk-vs-threshold

### Poza AUC: Różnicowanie niemal identycznych modeli

Kiedy wartości ROC-AUC są niemal identyczne (np. 0.84–0.85 dla różnych modeli), samo AUC nie może kierować wyborem modelu.

**TODO**: Wymień co najmniej 3 kryteria wykraczające poza wyniki statystyczne (ROC-AUC), które powinny różnicować modele. Dla każdego kryterium krótko wyjaśnij, dlaczego ma ono znaczenie.

In [ ]:
# TWOJA ODPOWIEDŹ TUTAJ
# Wymień co najmniej 3 kryteria wykraczające poza AUC i wyjaśnij, dlaczego każde z nich ma znaczenie.
# Napisz swoją odpowiedź jako instrukcje print() lub komentarze.

## 8. Analiza sprawiedliwości

**Zadanie**: Przeanalizuj, jak każdy model radzi sobie w podgrupach ze względu na płeć (M i K).

Dla każdego modelu:
1. Podziel zbiór testowy według `gender_test` na podgrupy M i K
2. Oblicz recall i precision dla każdej podgrupy
3. Oblicz różnicę recall: `|recall_M - recall_K|`

Różnica recall > 0.05 między grupami budzi obawy — oznacza, że model systematycznie lepiej identyfikuje zagrożonych klientów w jednej grupie.

In [ ]:
# TWÓJ KOD TUTAJ
# Dla każdego modelu i każdej grupy płci (M, K):
#   mask = (gender_test.values == g)
#   Oblicz recall i precision na zamaskowanym podzbiorze
# Następnie oblicz różnicę recall |recall_M - recall_K| dla każdego modelu
# Wyświetl wyniki w uporządkowanym formacie
# Który model ma najmniejszą różnicę recall?

## 9. Ocena interpretowalności

**Zadanie**: Porównaj interpretowalność modeli:
- **LogReg**: Pokaż 10 najważniejszych współczynników (wg wartości bezwzględnej) wraz ze znakami
- **RF**: Pokaż 10 najważniejszych feature importance
- **GB**: Pokaż 10 najważniejszych feature importance

Stwórz wykresy słupkowe poziome obok siebie dla wszystkich trzech modeli.

Następnie uszereguj modele od najbardziej do najmniej interpretowalnego i uzasadnij swoje rozumowanie.

In [ ]:
# TWÓJ KOD TUTAJ
# Współczynniki LogReg: lr_model.coef_[0] z feature_names lub X_train.columns
# Ważność cech RF: rf_model.feature_importances_
# Ważność cech GB: gb.feature_importances_
#
# Wyświetl top 10 dla każdego modelu
# Stwórz wykresy słupkowe obok siebie (1 wiersz, 3 kolumny)
# Uszereguj modele od najbardziej do najmniej interpretowalnego

## 10. Rekomendacja końcowa

**Scenariusz**: Prezentujesz wyniki Katarzynie Nowak, wiceprezes ds. marketingu w MajsterPlus. Pyta: *"Dlaczego powinnam zaufać temu modelowi?"* Nie jest osobą techniczną — musi zrozumieć:

- **(a)** Który model rekomendujesz i dlaczego
- **(b)** Ile pieniędzy przyniesie
- **(c)** Czy traktuje wszystkich klientów sprawiedliwie
- **(d)** Jak wytłumaczyć go zarządowi

**TODO**: Napisz rekomendację, która odpowiada na wszystkie cztery pytania Katarzyny. Użyj konkretnych liczb z Twojej analizy (zysk, threshold, różnice recall, interpretowalność). Powinna to być merytoryczna, wieloakapitowa rekomendacja biznesowa.

In [ ]:
# TWÓJ KOD TUTAJ
# Napisz kompleksową rekomendację odpowiadającą na wszystkie 4 pytania Katarzyny.
# Użyj konkretnych liczb z Twojej analizy.
# Napisz jako instrukcje print() — to jest Twój końcowy produkt.

## Opcjonalnie: Wzbogacenie z PyCaret (nieoceniane)

Jeśli chcesz szybko przetestować więcej modeli, odkomentuj poniższy fragment PyCaret.
Ta sekcja nie jest oceniana.

In [ ]:
# Opcjonalne wzbogacenie PyCaret — odkomentuj, aby uruchomić
# !pip install pycaret
# from pycaret.classification import setup, compare_models, pull
# pycaret_df = pd.concat([X_train, y_train], axis=1)
# clf = setup(pycaret_df, target=y_train.name, session_id=42)
# best = compare_models(n_select=3)
# results = pull()
# print(results)
pass  # placeholder

---
*Koniec MP5 — Gratulacje z okazji ukończenia wszystkich 5 mini-projektów!*